In [ ]:
print ("hello world")

: 

In [1]:
print ("hello world")

hello world


In [5]:
import polars as pl

base = "https://omicidx.cancerdatasci.org/data/latest"

studies = (
    pl.scan_parquet(f"{base}/sradb/study.parquet")
    .filter(pl.col("study_type") == "Transcriptome Analysis")
    .select("study_accession", "study_title")
    .head(10)
    .collect()
)
studies

study_accession,study_title
str,str
"""DRP000003""","""Comprehensive identification a…"
"""DRP000004""","""Comprehensive identification a…"
"""DRP000005""","""Comprehensive identification a…"
"""DRP000006""","""Comprehensive identification a…"
"""DRP000007""","""Comprehensive identification a…"
"""DRP000008""","""Comprehensive identification a…"
"""DRP000011""","""Comprehensive analysis of cyto…"
"""DRP000012""","""Comprehensive analysis of cyto…"
"""DRP000013""","""Comprehensive analysis of cyto…"


In [ ]:
#schema of sample.parquet



In [15]:
samples=(
    pl.scan_parquet(f"{base}/sradb/sample.parquet")
    .filter(pl.col("taxon_id") == 9685))

samples.schema

/var/folders/95/0dpzvggs067bzcs3lq08gfkm0000gn/T/ipykernel_10810/1310770400.py:5: PerformanceWarning: Resolving the schema of a LazyFrame is a potentially expensive operation. Use `LazyFrame.collect_schema()` to get the schema without this warning.
  samples.schema


Schema([('sample_id', Int64),
        ('sample_alias', String),
        ('sample_accession', String),
        ('taxon_id', Int32),
        ('scientific_name', String),
        ('description', String),
        ('sample_attribute', String),
        ('biosample_accession', String),
        ('scientific_name_as_submitted', String)])

In [ ]:
# sra_accessions lives at the release root (main schema), not under sradb/
accessions = pl.scan_parquet(f"{base}/sra_accessions.parquet")

accessions.collect_schema()

In [16]:
wgs_studies = (
    pl.scan_parquet(f"{base}/sradb/study.parquet"))

wgs_studies.schema

/var/folders/95/0dpzvggs067bzcs3lq08gfkm0000gn/T/ipykernel_10810/2889810284.py:4: PerformanceWarning: Resolving the schema of a LazyFrame is a potentially expensive operation. Use `LazyFrame.collect_schema()` to get the schema without this warning.
  wgs_studies.schema


Schema([('study_id', Int64),
        ('study_alias', String),
        ('study_accession', String),
        ('study_title', String),
        ('study_type', String),
        ('study_abstract', String),
        ('broker_name', String),
        ('center_name', String),
        ('center_project_name', String),
        ('study_description', String),
        ('study_attribute', String),
        ('bioproject_accession', String),
        ('geo_accession', String),
        ('pubmed_ids', List(String))])

In [27]:
# Felis catus (NCBI taxon 9685) SRA runs from the latest OmicIDX release.
# run_with_study already joins run -> experiment -> sample -> study, so no joins needed.
# Lazy scan + filter/select lets polars push down the predicate and fetch only these columns.
FELIS_CATUS = 9685
CAT_GENOME_SIZE = 2.7e9  # approx. Felis catus genome size in bases
MIN_COVERAGE = 1.0  # minimum coverage to keep a run

cat_runs = (
    pl.scan_parquet(f"{base}/sradb/run_with_study.parquet")
    .filter(pl.col("taxon_id") == FELIS_CATUS)
    .filter(pl.col("library_strategy") == "WGS")
   
    .select(
        "run_accession", "experiment_accession", "sample_accession",
        "study_accession", "study_title", "bioproject",
        "library_strategy", "library_source", "library_layout",
        "platform", "instrument_model", "total_spots", "total_bases",
    )
    .with_columns(
        (pl.col("total_bases") / 1e9).alias("total_gb"),
        (pl.col("total_bases") / CAT_GENOME_SIZE).alias("est_coverage"),
    )

    .filter(pl.col("est_coverage") >= MIN_COVERAGE)
    .collect()
)

run_ids = cat_runs["run_accession"].to_list()
print(f"{cat_runs.height:,} runs across {cat_runs['study_accession'].n_unique()} studies")
cat_runs.head()

1,954 runs across 68 studies


run_accession,experiment_accession,sample_accession,study_accession,study_title,bioproject,library_strategy,library_source,library_layout,platform,instrument_model,total_spots,total_bases,total_gb,est_coverage
str,str,str,str,str,str,str,str,str,str,str,i64,i64,f64,f64
"""ERR10482366""","""ERX10002625""","""ERS14236838""","""ERP142393""","""Limited historical admixture b…","""PRJEB57412""","""WGS""","""GENOMIC""","""SINGLE""","""ILLUMINA""","""unspecified""",41129368,3722839628,3.72284,1.378829
"""ERR10494350""","""ERX10012594""","""ERS14236711""","""ERP142393""","""Limited historical admixture b…","""PRJEB57412""","""WGS""","""GENOMIC""","""PAIRED""","""ILLUMINA""","""Illumina HiSeq 2500""",47188918,9532161436,9.532161,3.53043
"""ERR10494351""","""ERX10012595""","""ERS14236711""","""ERP142393""","""Limited historical admixture b…","""PRJEB57412""","""WGS""","""GENOMIC""","""PAIRED""","""ILLUMINA""","""Illumina HiSeq 2500""",47839004,9663478808,9.663479,3.579066
"""ERR10494352""","""ERX10012596""","""ERS14236711""","""ERP142393""","""Limited historical admixture b…","""PRJEB57412""","""WGS""","""GENOMIC""","""PAIRED""","""ILLUMINA""","""Illumina HiSeq 2500""",47635062,9622282524,9.622283,3.563808
"""ERR10494353""","""ERX10012597""","""ERS14236711""","""ERP142393""","""Limited historical admixture b…","""PRJEB57412""","""WGS""","""GENOMIC""","""PAIRED""","""ILLUMINA""","""Illumina HiSeq 2500""",39396117,7958015634,7.958016,2.947413


In [ ]:
# 


